# Notebook 5: LUFT Macro-Quantum Audit

This notebook demonstrates **Josephson junction physics**, **macroscopic quantum tunneling (MQT)**, and **LUFT foam modulation mapping**.

## Overview
- Implements Josephson junction plasma frequency and phase dynamics
- Explores macroscopic quantum tunneling vs thermal activation
- Maps LUFT foam modulation parameter f → E_J or C
- Performs sensitivity sweeps for Γ vs I/I_c
- Includes a synthetic switching-data MLE example to recover f

## Dependencies
- numpy
- scipy
- matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import expon
import sys
sys.path.insert(0, '../src')
from collapse import (
    plasma_omega, plasma_frequency, phase_effective_mass,
    EJ_from_Ic, deltaU_from_gamma, wkb_exponent,
    gamma_quantum, gamma_thermal, crossover_temperature,
    HBAR, K_B, PHI_0, E_CHARGE
)

## Physical Constants

In [ ]:
print(f"Planck constant (reduced): ħ = {HBAR:.6e} J·s")
print(f"Boltzmann constant: k_B = {K_B:.6e} J/K")
print(f"Flux quantum: Φ₀ = {PHI_0:.6e} Wb")
print(f"Elementary charge: e = {E_CHARGE:.6e} C")

## Parameter Presets

We define three parameter regimes:
1. **Measurement-like**: Large junctions used in metrology
2. **Qubit-like**: Superconducting qubit parameters
3. **Classical**: Larger junctions in the classical regime

In [ ]:
# Parameter presets
presets = {
    'measurement': {
        'I_c': 20e-6,      # 20 μA critical current
        'C': 2e-15,        # 2 fF capacitance
        'gamma': 0.5,      # moderate barrier
        'T': 0.015,        # 15 mK temperature
        'description': 'Measurement-like junction (metrology)'
    },
    'qubit': {
        'I_c': 0.5e-6,     # 0.5 μA critical current
        'C': 0.05e-15,     # 50 aF capacitance
        'gamma': 0.3,      # lower barrier
        'T': 0.020,        # 20 mK temperature
        'description': 'Qubit-like junction'
    },
    'classical': {
        'I_c': 100e-6,     # 100 μA critical current
        'C': 10e-15,       # 10 fF capacitance
        'gamma': 0.7,      # higher barrier
        'T': 4.2,          # 4.2 K temperature
        'description': 'Classical junction'
    }
}

# Display presets
for name, params in presets.items():
    E_J = EJ_from_Ic(params['I_c'])
    f_p = plasma_frequency(E_J, params['C'])
    T_c = crossover_temperature(E_J, params['gamma'])
    print(f"\n{params['description']}:")
    print(f"  I_c = {params['I_c']*1e6:.2f} μA")
    print(f"  C = {params['C']*1e15:.2f} fF")
    print(f"  E_J = {E_J:.3e} J ({E_J/K_B:.2f} K)")
    print(f"  f_p = {f_p*1e-9:.2f} GHz")
    print(f"  γ = {params['gamma']:.2f}")
    print(f"  T_crossover ≈ {T_c:.3f} K")
    print(f"  Operating T = {params['T']:.3f} K")

## Escape Rate vs. Barrier Parameter

Plot the total escape rate Γ = Γ_MQT + Γ_TA as a function of the barrier parameter γ.

In [ ]:
# Choose a preset
preset = presets['qubit']
E_J = EJ_from_Ic(preset['I_c'])
C = preset['C']
T = preset['T']

# Sweep gamma
gamma_values = np.linspace(0.1, 0.9, 50)
Gamma_mqt = np.array([gamma_quantum(E_J, C, g) for g in gamma_values])
Gamma_ta = np.array([gamma_thermal(E_J, C, g, T) for g in gamma_values])
Gamma_total = Gamma_mqt + Gamma_ta

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(gamma_values, Gamma_mqt, 'b-', label='Γ_MQT (quantum)', linewidth=2)
ax.semilogy(gamma_values, Gamma_ta, 'r--', label='Γ_TA (thermal)', linewidth=2)
ax.semilogy(gamma_values, Gamma_total, 'k-', label='Γ_total', linewidth=2)
ax.set_xlabel('Barrier parameter γ', fontsize=12)
ax.set_ylabel('Escape rate Γ (Hz)', fontsize=12)
ax.set_title(f'{preset["description"]} at T = {T:.3f} K', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAt γ = {preset['gamma']:.2f}:")
idx = np.argmin(np.abs(gamma_values - preset['gamma']))
print(f"  Γ_MQT = {Gamma_mqt[idx]:.3e} Hz")
print(f"  Γ_TA = {Gamma_ta[idx]:.3e} Hz")
print(f"  Γ_total = {Gamma_total[idx]:.3e} Hz")

## LUFT Foam Modulation: f → E_J Mapping

Model how LUFT foam density modulation parameter f affects the Josephson energy.
We assume: E_J(f) = E_J₀ * (1 + f)

In [ ]:
# Foam modulation sweep
f_values = np.linspace(-0.3, 0.3, 30)
E_J_nominal = EJ_from_Ic(preset['I_c'])
gamma_fixed = 0.4

# Compute rates for different foam modulations
Gamma_f = []
for f in f_values:
    E_J_mod = E_J_nominal * (1 + f)
    rate_mqt = gamma_quantum(E_J_mod, C, gamma_fixed)
    rate_ta = gamma_thermal(E_J_mod, C, gamma_fixed, T)
    Gamma_f.append(rate_mqt + rate_ta)

Gamma_f = np.array(Gamma_f)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(f_values, E_J_nominal * (1 + f_values) / K_B, 'b-', linewidth=2)
ax1.set_xlabel('LUFT foam parameter f', fontsize=12)
ax1.set_ylabel('E_J / k_B (K)', fontsize=12)
ax1.set_title('Foam Modulation → E_J Mapping', fontsize=14)
ax1.grid(True, alpha=0.3)

ax2.semilogy(f_values, Gamma_f, 'g-', linewidth=2)
ax2.set_xlabel('LUFT foam parameter f', fontsize=12)
ax2.set_ylabel('Escape rate Γ (Hz)', fontsize=12)
ax2.set_title(f'Escape Rate vs. Foam Modulation (γ = {gamma_fixed})', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFoam modulation range: f ∈ [{f_values[0]:.2f}, {f_values[-1]:.2f}]")
print(f"E_J range: [{E_J_nominal * (1 + f_values[0]) / K_B:.2f}, {E_J_nominal * (1 + f_values[-1]) / K_B:.2f}] K")
print(f"Γ range: [{np.min(Gamma_f):.3e}, {np.max(Gamma_f):.3e}] Hz")

## Sensitivity Sweep: Γ vs. I/I_c

Vary the bias current ratio I/I_c and compute the escape rate.

In [ ]:
# Bias current sweep
I_ratio = np.linspace(0.1, 0.95, 40)
I_c = preset['I_c']
E_J_base = EJ_from_Ic(I_c)

# For each bias, compute effective barrier
# Simple model: γ_eff ≈ γ₀ * sqrt(1 - (I/I_c)²)
gamma_0 = 0.6
gamma_eff = gamma_0 * np.sqrt(1 - I_ratio**2)

# Compute rates
Gamma_bias = []
for g_eff in gamma_eff:
    if g_eff < 0.01:  # avoid numerical issues
        Gamma_bias.append(np.inf)  # essentially instantaneous escape
    else:
        rate = gamma_quantum(E_J_base, C, g_eff) + gamma_thermal(E_J_base, C, g_eff, T)
        Gamma_bias.append(rate)

Gamma_bias = np.array(Gamma_bias)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(I_ratio, Gamma_bias, 'm-', linewidth=2)
ax.set_xlabel('Bias current I / I_c', fontsize=12)
ax.set_ylabel('Escape rate Γ (Hz)', fontsize=12)
ax.set_title('Escape Rate vs. Bias Current', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBias range: I/I_c ∈ [{I_ratio[0]:.2f}, {I_ratio[-1]:.2f}]")
print(f"Effective γ range: [{np.min(gamma_eff):.3f}, {np.max(gamma_eff):.3f}]")

## Synthetic Switching Data MLE

Generate synthetic switching time data with a known foam modulation parameter f, then use maximum likelihood estimation (MLE) to recover f.

In [ ]:
# Generate synthetic data
np.random.seed(42)
f_true = 0.15  # true foam modulation
gamma_meas = 0.45
E_J_true = E_J_base * (1 + f_true)
Gamma_true = gamma_quantum(E_J_true, C, gamma_meas) + gamma_thermal(E_J_true, C, gamma_meas, T)

# Sample switching times (exponentially distributed)
n_samples = 200
switching_times = np.random.exponential(1.0 / Gamma_true, n_samples)

print(f"True foam modulation: f = {f_true:.3f}")
print(f"True escape rate: Γ = {Gamma_true:.3e} Hz")
print(f"Generated {n_samples} synthetic switching events")
print(f"Mean switching time: {np.mean(switching_times):.3e} s (expected: {1/Gamma_true:.3e} s)")

In [ ]:
# Define negative log-likelihood function
def neg_log_likelihood(f_candidate):
    """Negative log-likelihood for foam modulation parameter f."""
    if f_candidate < -0.5 or f_candidate > 0.5:  # bounds check
        return 1e10
    
    E_J_candidate = E_J_base * (1 + f_candidate)
    Gamma_candidate = gamma_quantum(E_J_candidate, C, gamma_meas) + gamma_thermal(E_J_candidate, C, gamma_meas, T)
    
    if Gamma_candidate <= 0:
        return 1e10
    
    # Exponential likelihood: L = prod(Γ * exp(-Γ * t_i))
    # Negative log-likelihood: -log(L) = -n*log(Γ) + Γ*sum(t_i)
    nll = -n_samples * np.log(Gamma_candidate) + Gamma_candidate * np.sum(switching_times)
    return nll

# Perform MLE
result = minimize(neg_log_likelihood, x0=0.0, method='Nelder-Mead')
f_mle = result.x[0]

print(f"\nMLE Results:")
print(f"Estimated foam modulation: f_MLE = {f_mle:.3f}")
print(f"True value: f_true = {f_true:.3f}")
print(f"Relative error: {100 * abs(f_mle - f_true) / f_true:.1f}%")

E_J_mle = E_J_base * (1 + f_mle)
Gamma_mle = gamma_quantum(E_J_mle, C, gamma_meas) + gamma_thermal(E_J_mle, C, gamma_meas, T)
print(f"\nEstimated escape rate: Γ_MLE = {Gamma_mle:.3e} Hz")
print(f"True escape rate: Γ_true = {Gamma_true:.3e} Hz")

In [ ]:
# Plot likelihood landscape
f_scan = np.linspace(-0.3, 0.5, 100)
nll_scan = [neg_log_likelihood(f) for f in f_scan]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Likelihood landscape
ax1.plot(f_scan, nll_scan, 'b-', linewidth=2)
ax1.axvline(f_true, color='g', linestyle='--', linewidth=2, label=f'True: f = {f_true:.3f}')
ax1.axvline(f_mle, color='r', linestyle='--', linewidth=2, label=f'MLE: f = {f_mle:.3f}')
ax1.set_xlabel('Foam modulation parameter f', fontsize=12)
ax1.set_ylabel('Negative log-likelihood', fontsize=12)
ax1.set_title('MLE Likelihood Landscape', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Switching time histogram
ax2.hist(switching_times, bins=30, density=True, alpha=0.6, color='blue', label='Data')
t_theory = np.linspace(0, np.max(switching_times), 100)
pdf_true = Gamma_true * np.exp(-Gamma_true * t_theory)
pdf_mle = Gamma_mle * np.exp(-Gamma_mle * t_theory)
ax2.plot(t_theory, pdf_true, 'g-', linewidth=2, label='True PDF')
ax2.plot(t_theory, pdf_mle, 'r--', linewidth=2, label='MLE PDF')
ax2.set_xlabel('Switching time (s)', fontsize=12)
ax2.set_ylabel('Probability density', fontsize=12)
ax2.set_title('Switching Time Distribution', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. Josephson junction physics and plasma frequency calculations
2. Macroscopic quantum tunneling vs. thermal activation regimes
3. LUFT foam modulation mapping (f → E_J)
4. Sensitivity of escape rates to junction parameters
5. Maximum likelihood estimation to recover foam modulation from switching data

The MLE analysis successfully recovered the foam modulation parameter f with good accuracy, demonstrating the feasibility of auditing LUFT-induced quantum effects in Josephson junctions.